# Django, ModelForms

## Introduction to ModelForms
---

In this lesson, you'll learn how to **create forms directly from your Django models**.

`ModelForms` eliminate duplication by automatically generating form fields based on your model's field definitions.

1. [Why ModelForms](#Why-ModelForms),
    - [DRY Principle](#DRY-Principle),
    - [ModelForm vs Form](#ModelForm-vs-Form),
2. [Creating a ModelForm](#Creating-a-ModelForm),
    - [Basic Structure](#Basic-Structure),
    - [The Meta Class](#The-Meta-Class),
3. [Controlling Fields](#Controlling-Fields),
    - [fields vs exclude](#fields-vs-exclude),
    - [Field Customization](#Field-Customization),
4. [Saving Data](#Saving-Data),
    - [The save() Method](#The-save()-Method),
    - [commit=False Pattern](#commit=False-Pattern),
5. [Editing Existing Objects](#Editing-Existing-Objects),
    - [The instance Parameter](#The-instance-Parameter),
    - [Update Views](#Update-Views),
6. [🧠 EXERCISE 🧠, Create a Blog ModelForm](#🧠-EXERCISE-🧠,-Create-a-Blog-ModelForm),
7. [🧠 EXERCISE 🧠, Build a Blog Edit View](#🧠-EXERCISE-🧠,-Build-a-Blog-Edit-View).

<br>

## Why `ModelForms`

---

### DRY Principle

---

Consider this model and its corresponding form:

<br>

**The Problem - Duplication:**

Take a look at the following scenario. Below, we have two cells where the model class and the form class share the same attributes.

In [ ]:
# blog/models.py - Model definition (current project)
from django.db import models


class Blog(models.Model):
    title = models.CharField(max_length=200)
    author = models.CharField(max_length=100)
    author_email = models.EmailField(blank=True)
    content = models.TextField(blank=True)
    published_date = models.DateField()

In [ ]:
# blog/forms.py - Manual form definition (BAD - duplicates model fields!)
from django import forms


class BlogCreateForm(forms.Form):  # BAD: Duplicating model fields
    title = forms.CharField(max_length=200)
    author = forms.CharField(max_length=100)
    author_email = forms.EmailField(required=False)
    content = forms.CharField(required=False, widget=forms.Textarea)
    published_date = forms.DateField()

<br>

**The problems with this approach:**

| Issue | Consequence |
|-------|-------------|
| Duplicated field definitions | Changes need to be made in two places |
| Manual validation matching | Easy to get out of sync |
| Manual save logic | You write `book.title = form.cleaned_data['title']` for every field |
| Error-prone | Typos, forgotten fields, inconsistent constraints |

<br>

### ModelForm vs Form

---

**ModelForm** solves this by generating form fields from your model:

<br>

| Feature | `forms.Form` | `forms.ModelForm` |
|---------|--------------|-------------------|
| Field source | Manually defined | Auto-generated from model |
| Validation | Manual | Inherits model constraints |
| Saving | Manual `Model.objects.create()` | Built-in `form.save()` |
| Use case | Any data collection | CRUD operations on models |

<br>

**Use `ModelForm` when:**
- You're creating/editing model instances
- Form fields match model fields closely

**Use regular `Form` when:**
- Data doesn't map to a model (e.g., search, contact forms)
- Form structure differs significantly from any model
- Use a form only when you need to validate specific attributes of a model."

<br>

## Creating a `ModelForm`

---

### Basic Structure

---

A `ModelForm` is a form class that inherits from `forms.ModelForm`:

In [ ]:
# blog/forms.py
from django import forms
from blog.models import Blog


class BlogModelForm(forms.ModelForm):  # GOOD: Inherits from ModelForm
    """Form for creating and editing blogs."""

    class Meta:
        model = Blog
        fields = ['title', 'author', 'author_email', 'content', 'published_date', 'category_type']

In [ ]:
# EXAMPLE: Run inside the shell
from blog.forms import BlogModelForm

# Happy-case
correct_blog_data = {
    'title': 'Test blog, new version',
    'author': 'matous_holinka',
    'author_email': 'matous@example.com',
    'content': 'Blog content here...',
    'published_date': '2026-04-14',
    'category_type': 1,  # CategoryType.TECH
}
form = BlogModelForm(data=correct_blog_data)
form.is_valid(), form.errors

# Error-case (missing required fields)
incorrect_blog_data = {
    'title': 'Test blog',
    'author': 'matous_holinka',
}
form_bad = BlogModelForm(data=incorrect_blog_data)
form_bad.is_valid(), form_bad.errors
# {'published_date': ['This field is required.'], 'category_type': ['This field is required.']}

<br>

That's it! Django automatically:
- Creates form fields matching the model fields
- Applies the same constraints (`max_length`, `blank`, etc.)
- Provides a `save()` method that creates/updates the model instance

<br>

### The Meta Class

---

The `Meta` inner class configures how the ModelForm behaves:

<br>

| Attribute | Purpose | Example |
|-----------|---------|--------|
| `model` | Which model to use | `model = Blog` |
| `fields` | Which fields to include | `fields = ['title', 'author']` |
| `exclude` | Which fields to exclude | `exclude = ['slug']` |
| `widgets` | Custom widgets for fields | `widgets = {'content': Textarea}` |
| `labels` | Custom labels | `labels = {'author_email': 'Author E-mail'}` |
| `help_texts` | Custom help texts | `help_texts = {'status': 'draft/published/archived'}` |
| `error_messages` | Custom error messages | `error_messages = {...}` |

In [ ]:
# EXAMPLE: Fully configured Meta class
from django import forms
from blog.models import Blog


class BlogModelForm(forms.ModelForm):
    """Form for creating and editing blogs."""

    class Meta:
        model = Blog
        fields = ['title', 'author', 'author_email', 'content', 'published_date', 'category_type']

        widgets = {
            'published_date': forms.DateInput(
                attrs={'type': 'date'}  # HTML5 date picker
            ),
            'content': forms.Textarea(
                attrs={'rows': 4, 'placeholder': 'Enter blog content...'}
            ),
        }

        labels = {
            'author_email': 'Author E-mail',
            'published_date': 'Publication Date',
        }

        help_texts = {
            'category_type': 'Choose the category that best fits your post.',
        }

Typically, we want to define two main attributes: the `model` and the `fields` we want to work with. Everything else is essentially optional.

<br>

## Controlling Fields

---

### `fields` vs. `exclude`

---

You **must** specify which fields to include in the form. There are three approaches:

<br>

**1. Explicit field list (recommended):**

In [ ]:
class BlogModelForm(forms.ModelForm):
    class Meta:
        model = Blog
        fields = ['title', 'author', 'published_date']  # Only these fields

<br>

**2. All fields:**

In [ ]:
class BlogModelForm(forms.ModelForm):
    class Meta:
        model = Blog
        fields = '__all__'  # Insecure: All model fields (be careful!)

<br>

**3. Exclude specific fields:**

In [ ]:
class BlogModelForm(forms.ModelForm):
    class Meta:
        model = Blog
        exclude = ['slug']  # All except these

<br>

**Security Warning:** Always use explicit `fields` when possible!

| Approach | Security | Recommendation |
|----------|----------|----------------|
| `fields = [...]` | ✅ Safe | **Recommended** - explicit is better |
| `fields = '__all__'` | ⚠️ Risky | Avoid - exposes all fields |
| `exclude = [...]` | ⚠️ Risky | Avoid - new fields auto-included |

Using `exclude` or `__all__` can accidentally expose sensitive fields when you add new fields to the model.

<br>

### Field Customization

---

You can override or add fields to a `ModelForm` just like a regular form:

In [ ]:
from django import forms
from blog.models import Blog, CategoryType


class BlogModelForm(forms.ModelForm):
    """Form with customised and additional fields."""

    # Override an existing field — add help text to category_type
    category_type = forms.ChoiceField(
        choices=CategoryType.choices,
        help_text='Select the category that best fits this post.'
    )

    # Add a field that does not exist in the model (handle in view or clean())
    confirm_author = forms.CharField(
        label='Confirm Author',
        help_text='Re-enter author name to confirm.'
    )

    class Meta:
        model = Blog
        fields = ['title', 'author', 'author_email', 'content', 'published_date', 'category_type']

**The Model:** Defines how data is stored in the database (e.g., `category_type` is an integer).

**The Form:** Defines how the user interacts with that data (e.g., explaining what `'Category Type'` means).

**The Key Advantage:** Overriding fields in a `ModelForm` allows you to improve the User Experience (UX)—by adding `help_text` or custom labels—without affecting your database structure."

<br>

🔑 **Field override rules:**
- If you define a field with the same name as a model field, your definition takes precedence
- You can add fields that don't exist in the model (handle them in `clean()` or the view)
- Model field constraints still apply during save (database-level)

<br>

## Saving Data

---

### The `save()` Method

---

ModelForm provides a `save()` method that creates or updates the model instance:

In [ ]:
# blog/views.py - HTML create view (ModelForm)

from django.shortcuts import render, redirect
from blog.forms import BlogModelForm

from django.views.decorators.csrf import csrf_exempt
from django.http import JsonResponse
import json


@csrf_exempt  # Only for demo/curl testing - in production use CSRF tokens
def blog_create(request):
    form = BlogModelForm(request.POST or None)  # TODO: Explain

    if form.is_valid():
        try:
            blog = form.save()
        except (DatabaseError, IntegrityError):
            return render(request, 'blog/blog_form.html', {'form': form, 'error': 'Nepodařilo se uložit blog. Zkuste to znovu.'})
        return redirect('blog:blog_detail', id=blog.id)
    return render(request, 'blog/blog_form.html', {'form': form})


# curl happy-case:
curl -X POST http://localhost:8000/blog/create/ \
  -H "Content-Type: application/json" \
  -d '{"title": "My Blog", "author": "John Doe", "published_date": "2024-01-01", "category_type": 1}'

# curl error-case (invalid author — contains special char, missing published_date → 400):
curl -X POST http://localhost:8000/blog/create/ \
  -H "Content-Type: application/json" \
  -d '{"title": "My Blog", "author": "john_doe!"}'

In [ ]:
<!-- blog/templates/blog/blog_form.html -->
{% extends "blog/base.html" %}

{% block title %}
  {% if blog %}Edit Blog{% else %}Create Blog{% endif %}
{% endblock %}

{% block content %}
  <h1>{% if blog %}Edit blog{% else %}Create a new blog{% endif %}</h1>

  <form method="post">
    {% csrf_token %}
    {{ form.as_p }}
    <button type="submit">{% if blog %}Update{% else %}Save{% endif %}</button>
  </form>
{% endblock %}

**Project note (important):** In this repo, `blog.urls` uses `app_name = "blog"`, so use namespaced redirects like `redirect("blog:blog_list")` in all create/update/delete examples.

<br>

**What `save()` does:**

1. Creates a model instance with the form data
2. Calls `instance.save()` to persist to the database
3. Returns the saved instance

<br>

### `commit=False` pattern

---

Sometimes you need to modify the instance before saving. Use `commit=False`:

In [ ]:
# Adding data not in the form before saving

def blog_create(request):
    """View to create a blog with additional data."""

    if request.method == 'POST':
        form = BlogModelForm(request.POST)

        if form.is_valid():
            # commit=False creates instance but doesn't save to database
            blog = form.save(commit=False)

            # Just simple DEMO
            # you should always generate slug in the model (w/ slugify())
            if not blog.slug:
                blog.slug = blog.title.lower().replace(' ', '-')
            blog.status = 'draft'

            # Now save to database
            blog.save()

            return redirect('blog:blog_list')
    else:
        form = BlogModelForm()

    return render(request, 'blog/blog_form.html', {'form': form})

<br>

**Common uses for `commit=False`:**

| Scenario | Example |
|----------|--------|
| Add user reference | `blog.created_by = request.user` |
| Set timestamps | `blog.created_at = timezone.now()` |
| Set computed values | `blog.slug = slugify(blog.title)` |
| Set defaults | `blog.status = 'draft'` |

<br>

**Important:** If your model has `ManyToMany` fields, you must call `save_m2m()` after using `commit=False`:

In [ ]:
# Example: commit=False + save_m2m pattern (generic example)

# NOTE: Current Blog model in this project has no ManyToMany fields.
# Keep this as a reusable pattern for models that do.

def create_object_with_m2m(request):
    if request.method == 'POST':
        form = SomeModelForm(request.POST)
        if form.is_valid():
            obj = form.save(commit=False)
            obj.save()
            form.save_m2m()
            return redirect('success_url')
    else:
        form = SomeModelForm()

    return render(request, 'some_template.html', {'form': form})

🔑 When you use `commit=False`, Django "steps out" of the automatic saving process. Because the form didn't save the instance to the database immediately, it lost the ability to automatically save those join tables.

First, you must save the instance manually (`instance.save()`), which generates its ID.

Then, you must tell the form: `"Hey, the instance is in the DB now, so go ahead and fill in those Many-to-Many relationships."`

That is exactly what `save_m2m()` is for.

<br>

## Editing Existing Objects

---

### The instance Parameter

---

To edit an existing object, pass it as the `instance` parameter:

In [ ]:
# blog/views.py - Edit view (ModelForm)
from django.shortcuts import render, redirect, get_object_or_404
from blog.models import Blog
from blog.forms import BlogModelForm


def blog_update(request: HttpRequest, pk: int):
    blog = get_object_or_404(Blog, pk=pk)

    if request.method == 'POST':
        form = BlogModelForm(request.POST, instance=blog)
        if form.is_valid():
            form.save()  # UPDATE — same save(), Django issues UPDATE SQL instead of INSERT
            return redirect('blog:blog_list')
    else:
        form = BlogModelForm(instance=blog)  # pre-fill form with existing data

    return render(request, 'blog/blog_form.html', {'form': form, 'blog': blog})

In [ ]:
# blog/urls.py  — actual URL configuration in this project  
from django.urls import path
from blog import views

app_name = 'blog'

urlpatterns = [
    # ...
    path('<int:pk>/edit/', views.blog_update, name='blog_update'),
    # ...
]

<br>

**How `instance` works:**

| Without `instance` | With `instance` |
|--------------------|----------------|
| Form fields are empty | Form fields are pre-filled |
| `save()` creates new object | `save()` updates existing object |
| `INSERT` SQL | `UPDATE` SQL |

<br>

### Update Views

---

A complete pattern for create and update views:

In [ ]:
# blog/views.py - Complete CRUD views

from django.shortcuts import render, redirect, get_object_or_404
from django.contrib import messages
from .models import Blog
from .forms import BlogModelForm


@csrf_exempt
@require_http_methods(["GET", "POST"])
def blog_create(request: HttpRequest) -> HttpResponse:
    form = BlogModelForm(request.POST or None)

    if form.is_valid():
        try:
            blog = form.save(commit=False)

            if not blog.slug:
                blog.slug = blog.title.lower().replace(' ', '-')
            blog.save()

        except (DatabaseError, IntegrityError):
            return render(request, 'blog/blog_form.html', {'form': form, 'error': 'Nepodařilo se uložit blog. Zkuste to znovu.'})
        return redirect('blog:blog_detail', id=blog.id)
    return render(request, 'blog/blog_form.html', {'form': form})


@csrf_exempt
def blog_update(request: HttpRequest, pk: int) -> HttpResponse:
    blog = get_object_or_404(Blog, pk=pk)

    if request.method == 'POST':
        form = BlogModelForm(request.POST, instance=blog)
        if form.is_valid():
            form.save()
            return render(request, 'blog/blog_update_success.html', {'form': form, 'blog': blog})
    else:
        form = BlogModelForm(instance=blog)  # pre-fill form with existing data

    return render(request, 'blog/blog_form.html', {'form': form, 'blog': blog})

In [ ]:
# templates/blog/blog_form.html - Reusable template
"""
<!-- blog/templates/blog/blog_form.html -->
{% extends "blog/base.html" %}

{% block title %}
  {% if blog %}Edit Blog{% else %}Create Blog{% endif %}
{% endblock %}

{% block content %}
  <h1>{% if blog %}Edit blog{% else %}Create a new blog{% endif %}</h1>

  <form method="post">
    {% csrf_token %}
    {{ form.as_p }}
    <button type="submit">{% if blog %}Update{% else %}Save{% endif %}</button>
  </form>
{% endblock %}
"""

In [ ]:
# blog/urls.py  — actual URL configuration in this project

from django.urls import path
from blog import views  # correct import

app_name = 'blog'

urlpatterns = [
    # === HTML views (classic Django application) ===
    path('', views.blog_list, name='blog_list'),
    path('<int:id>/', views.BlogDetailView.as_view(), name='blog_detail'),
    path('create/', views.blog_create, name='blog_create'),
    path('<int:pk>/edit/', views.blog_update, name='blog_update'),
    path('search/', views.blog_search, name='blog_search'),
    path('comment/create/', views.comment_create, name='comment_create'),
    path('review-form/', views.review_create, name='review_form'),
    path('review-form/success/', views.review_success, name='review_form_success'),

    # === JSON API views (REST-like approach) ===
    path('api/blogs/', views.api_blog_list_create, name='api_blog_list_create'),
    path('api/comments/', views.api_comment_create, name='api_comment_create'),
]

<br>

## Combining Create and Update

---

🔑 You can combine create and update into a single view:

In [ ]:
# Combined create/update view

def blog_form(request, pk=None):
    """Create or update a blog depending on whether pk is provided."""

    # Get existing blog or None for new blog
    if pk:
        blog = get_object_or_404(Blog, pk=pk)
        title = f'Edit: {blog.title}'
    else:
        blog = None
        title = 'Add New Blog'

    if request.method == 'POST':
        form = BlogModelForm(request.POST, instance=blog)
        if form.is_valid():
            saved_blog = form.save()
            action = 'updated' if pk else 'created'
            messages.success(request, f'Blog {action} successfully!')
            return redirect('blog_list')
    else:
        form = BlogModelForm(instance=blog)

    return render(request, 'blog/blog_form.html', {
        'form': form,
        'blog': blog,
        'title': title
    })

In [ ]:
# URL patterns for combined view

urlpatterns = [
    # ...
    path('blogs/new/', views.blog_form, name='blog_create'),
    path('blogs/<int:pk>/edit/', views.blog_form, name='blog_update'),
    # ...
]

<br>

## Summary

---

| Concept | Key Points |
|---------|------------|
| **ModelForm** | Form generated from model - follows DRY principle |
| **Meta class** | Configure model, fields, widgets, labels |
| **fields** | Always use explicit list for security |
| **exclude** | Avoid - can accidentally expose new fields |
| **save()** | Creates or updates model instance |
| **commit=False** | Create instance without saving - modify first |
| **save_m2m()** | Required after commit=False with ManyToMany fields |
| **instance** | Pass existing object to edit instead of create |

<br>

### 🧠 EXERCISE 🧠, Create a Blog ModelForm

---

Create a ModelForm for managing blogs in your current `blog` app:

1. Use existing `Blog` model in `blog/models.py` (already present in project).

2. Create `BlogModelForm` in `blog/forms.py`:
   - Include only explicit safe fields (no `__all__`)
   - Use a date input widget for `published_date`
   - Add labels/help text for at least one field (`status` or `author_email`)

3. Update existing `blog_create` view so it uses `BlogModelForm` and `form.save()`.

4. Add/update template and URL pattern for blog create.

5. Test valid and invalid submissions (including date/status validation).

<details>
    <summary>▶️ Solution</summary>

**1. blog/forms.py:**

```python
from django import forms
from .models import Blog


class BlogModelForm(forms.ModelForm):
    class Meta:
        model = Blog
        fields = ['title', 'author', 'author_email', 'content', 'published_date', 'status']
        widgets = {
            'published_date': forms.DateInput(attrs={'type': 'date'}),
            'content': forms.Textarea(attrs={'rows': 4}),
        }
        labels = {
            'author_email': 'Author E-mail',
            'published_date': 'Publication Date',
        }
        help_texts = {
            'status': 'Allowed values: draft, published, archived.',
        }
```

**2. blog/views.py (update existing view):**

```python
from django.shortcuts import render, redirect
from .forms import BlogModelForm


def blog_create(request):
    if request.method == 'POST':
        form = BlogModelForm(request.POST)
        if form.is_valid():
            form.save()
            return redirect('blog_list')
    else:
        form = BlogModelForm()

    return render(request, 'blog/blog_form.html', {'form': form})
```

**3. templates/blog/blog_form.html:**

```html
<form method="post">
    {% csrf_token %}
    {{ form.as_p }}
    <button type="submit">Save Blog</button>
</form>
```

**4. blog/urls.py:**

```python
from django.urls import path
from . import views

urlpatterns = [
    path('blogs/new/', views.blog_create, name='blog_create'),
]
```

**5. Test:**

```bash
python manage.py runserver
# Visit http://127.0.0.1:8000/blogs/new/
```
</details>

<br>

### 🧠 EXERCISE 🧠, Build a Blog Edit View

---

Extend the previous exercise to support editing:

1. Create a `blog_update` view that:
   - Takes a `pk` parameter to identify the blog
   - Pre-fills the form with existing data
   - Updates the blog on valid submission

2. Add a URL pattern for the edit view

3. Modify the template to show "Edit" or "Create" based on context

4. Create a blog via the create form, then edit it

<details>
    <summary>▶️ Solution</summary>

**1. blog/views.py (add update view):**

```python
from django.shortcuts import render, redirect, get_object_or_404
from .models import Blog
from .forms import BlogModelForm


def blog_update(request, pk):
    blog = get_object_or_404(Blog, pk=pk)

    if request.method == 'POST':
        form = BlogModelForm(request.POST, instance=blog)
        if form.is_valid():
            form.save()
            return redirect('blog_list')
    else:
        form = BlogModelForm(instance=blog)

    return render(request, 'blog/blog_form.html', {
        'form': form,
        'blog': blog,
        'title': f'Edit: {blog.title}'
    })
```

**2. blog/urls.py:**

```python
from django.urls import path
from . import views

urlpatterns = [
    path('blogs/new/', views.blog_create, name='blog_create'),
    path('blogs/<int:pk>/edit/', views.blog_update, name='blog_update'),
]
```

**3. templates/blog/blog_form.html:**

```html
<form method="post">
    {% csrf_token %}
    {{ form.as_p }}
    <button type="submit">
        {% if blog %}Update{% else %}Create{% endif %} Blog
    </button>
</form>
```

**4. Test:**

```bash
python manage.py runserver
# Create: http://127.0.0.1:8000/blogs/new/
# Edit: http://127.0.0.1:8000/blogs/1/edit/
```
</details>

---